# Arm E: EyePACS only, on the group-aware split's EyePACS data
**Why:** the planned comparison is **A = DDR only, B = EyePACS only (this arm, called arm E in the code), C = full DDR + EyePACS merge**, each tested on unseen APTOS and on each source's held-out test split. One validation rule for every arm: the best epoch, calibration and threshold are chosen on a **held-out part of the arm's own training data** (for arm E, the held-out EyePACS validation patients).

Trains arm E for seeds 42, 43, 44 (EfficientNet-B0, 512 px, mild augmentation, 10 epochs) on the same frozen splits as the other arms (**training only: no test set is opened**; evaluation is the separate notebook `evaluate_arm_E`). About **3.5 hours** on a GPU for three seeds (about 1 hour per seed once the image cache is warm, plus a slower first epoch); finished runs are skipped if you rerun, so it can be done over several sessions.

**Before running**
* Accelerator **GPU**, Internet **On** (ImageNet weights).
* Add as inputs: (1) the output of your original run (processed images, `manifest.csv`), (2) the output of **`ddr_group_split`** (`audit/ddr_splits_all.csv`; it fixes the DDR test split used later). No code dataset is needed.
* After training, **Save Version -> Save & Run All (Commit)** so the models are kept, then run `evaluate_arm_E`.

**What to send me:** the validation table at the end.

In [ ]:
import base64, glob, json, os, shutil, subprocess, sys, zlib
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
CODE = os.path.join(WORK, "code")
EXTRA = os.environ.get("EARLYDR_TEST", "").split()     # test hook, leave empty on Kaggle
CODE_FILES = (
    "..."
)
try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"])
for name, text in json.loads(zlib.decompress(base64.b64decode(CODE_FILES))).items():
    path = os.path.join(CODE, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
print("code written to", CODE)
sys.path.insert(0, CODE)

def real_dirs(pattern):
    return [os.path.dirname(f) for f in glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)]

def processed_in(folder):
    return [p for p in glob.glob(os.path.join(folder, "processed_*")) if os.path.isdir(p) and not os.path.islink(p)]

# build the v2 work folder: original manifest + the group split for DDR + the original processed images
data_dirs = [d for d in real_dirs("manifest.csv") if processed_in(d)]
split_files = glob.glob(os.path.join(INPUT, "**", "audit", "ddr_splits_all.csv"), recursive=True)
if not data_dirs:
    sys.exit("STOP: add the output of your original run (it has manifest.csv and the processed images) as an input.")
if not split_files:
    sys.exit("STOP: add the output of the ddr_group_split notebook (it has audit/ddr_splits_all.csv) as an input.")
DATA, SPLITS = data_dirs[0], split_files[0]
print("images and manifest from:", DATA, "\nsplits from:", SPLITS)

import pandas as pd
V2 = os.path.join(WORK, "v2")
os.makedirs(V2, exist_ok=True)
manifest = pd.read_csv(os.path.join(DATA, "manifest.csv"))
groups = pd.read_csv(SPLITS).set_index("image_id")["group"]
is_ddr = manifest.source == "DDR"
assert manifest[is_ddr].image_id.isin(groups.index).all(), "split file does not cover every DDR image"
manifest.loc[is_ddr, "split"] = manifest.loc[is_ddr, "image_id"].map(groups).values
manifest.to_csv(os.path.join(V2, "manifest.csv"), index=False)
manifest.to_csv(os.path.join(V2, "splits.csv"), index=False)
processed = processed_in(DATA)[0]
link = os.path.join(V2, os.path.basename(processed))
if not os.path.exists(link):
    os.symlink(processed, link)
print(manifest.groupby(["source", "split"]).size().unstack(fill_value=0))

## Train arm E (3 seeds)

In [ ]:
SEEDS = [42, 43, 44]
EPOCHS = 10
subprocess.run([sys.executable, "-m", "early_dr.run_experiment", "configs/arms/arm_E.yaml", "--set",
                "task=stage1", f"paths.work={V2}", f"train.epochs={EPOCHS}", *EXTRA,
                "--stage", "train", "--seeds", *map(str, SEEDS)], cwd=CODE)
rows = []
for f in sorted(glob.glob(os.path.join(V2, "runs", "arm_E", "seed*", "train_info.json"))):
    info = json.load(open(f))
    rows.append({"run": "arm_E", "seed": f.split(os.sep)[-2], "best validation PR-AUC": round(info["best_val_score"], 4), "epochs": len(info["history"])})
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe validation set of arm E is the held-out EyePACS patients only, so this score is comparable with the EyePACS-only validation scores, not with the arms that also validate on DDR.")